# V5 Model Training (HANDOFF #2)

**Expected runtime:** 1-3 hours on Colab Pro high-RAM CPU.

**What this trains:** 54 ensembles = 27 (position, stat) pairs × 2 model types (StatPredictor + POBModel).
Per-position algorithm subsets — total 174 `.joblib` files + 54 `_meta.json`.

| Position | Algorithms | Stats | Files |
|----------|-----------|-------|-------|
| QB  | xgboost, lightgbm, catboost | 5 | 30 |
| RB  | xgboost, lightgbm, catboost, random_forest | 5 | 40 |
| WR  | xgboost, lightgbm, catboost | 4 | 24 |
| TE  | xgboost, lightgbm, catboost, random_forest | 4 | 32 |
| K   | xgboost, random_forest | 3 | 12 |
| DST | xgboost, catboost, random_forest | 6 | 36 |

**Resumable:** Per-ensemble. If all algo `.joblib` + meta JSON exist for a (pos, stat, type), it is skipped.

## Drive structure required

```
My Drive/SportsAnalyzer/
├── src/nfl/training/v5/         ← V5 training package (upload from local repo)
│   ├── config.py
│   ├── data.py
│   ├── models.py
│   ├── walkforward.py
│   └── train.py
├── src/nfl/features/v5/         ← features V5 (already uploaded for HANDOFF #1.5)
├── output/features/v5/          ← 16 parquets from HANDOFF #1.5
└── output/models/v5/            ← this notebook writes here (174 .joblib + 54 .json + _mae_summary.csv)
```

## Step 1 — Mount Drive

In [ ]:
print('[Step 1/8] Mounting Google Drive...')
from google.colab import drive
drive.mount('/content/drive')
print('[Step 1/8] Drive mounted')

## Step 2 — Set paths and verify code uploaded

In [ ]:
print('[Step 2/8] Setting paths and verifying code...')
import sys
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/SportsAnalyzer'
FEATURES_DIR = f'{DRIVE_ROOT}/output/features/v5'
MODELS_DIR = f'{DRIVE_ROOT}/output/models/v5'
CODE_ROOT = DRIVE_ROOT

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

assert Path(FEATURES_DIR).exists(), f'ERROR: Features dir not found: {FEATURES_DIR} — run v5_feature_engineering.ipynb first'
assert Path(f'{CODE_ROOT}/src/nfl/training/v5/train.py').exists(), \
    f'ERROR: V5 training code not uploaded. Copy src/nfl/training/v5/ to {CODE_ROOT}/src/nfl/training/v5/'

# Pre-flight: verify both player and DST parquets are present (8 of each expected).
player_files = sorted(Path(FEATURES_DIR).glob('features_2*.parquet'))
dst_files = sorted(Path(FEATURES_DIR).glob('features_dst_*.parquet'))
# features_2*.parquet matches both player and dst parquets — narrow to player only
player_only = [p for p in player_files if 'features_dst_' not in p.name]
assert len(player_only) >= 6, \
    f'ERROR: Expected 8 player parquets, found {len(player_only)} in {FEATURES_DIR}'
assert len(dst_files) >= 6, \
    f'ERROR: Expected 8 DST parquets (features_dst_*), found {len(dst_files)} in {FEATURES_DIR}. ' \
    f'Re-run v5_feature_engineering.ipynb Step 5b to build them.'

for init_path in [
    f'{CODE_ROOT}/src/__init__.py',
    f'{CODE_ROOT}/src/nfl/__init__.py',
    f'{CODE_ROOT}/src/nfl/training/__init__.py',
]:
    Path(init_path).touch(exist_ok=True)

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

print(f'[Step 2/8] Paths verified')
print(f'  Features: {FEATURES_DIR}  ({len(player_only)} player + {len(dst_files)} DST parquets)')
print(f'  Code:     {CODE_ROOT}/src/nfl/training/v5/')
print(f'  Models:   {MODELS_DIR}')

## Step 3 — High-RAM / CPU sanity check

Training 174 models needs sustained memory. If this cell warns, switch to High-RAM runtime.

In [ ]:
print('[Step 3/8] Checking RAM and CPU...')
import psutil, os
ram_gb = psutil.virtual_memory().total / 1e9
cpu_count = os.cpu_count()
print(f'  RAM:  {ram_gb:.1f} GB')
print(f'  CPUs: {cpu_count}')
if ram_gb < 20:
    print('\n  WARNING: Not using high-RAM runtime.')
    raise RuntimeError('Aborting — switch to high-RAM runtime first')
print('[Step 3/8] Runtime looks good')

## Step 4 — Install ML libraries (if missing)

In [ ]:
print('[Step 4/8] Installing/verifying ML libs...')
import importlib
for lib, pip_name in [('xgboost','xgboost'),('lightgbm','lightgbm'),('catboost','catboost')]:
    try:
        importlib.import_module(lib)
        print(f'  OK {lib}')
    except ImportError:
        print(f'  Installing {pip_name}...')
        import subprocess
        subprocess.run(['pip','install','-q',pip_name], check=True)
print('[Step 4/8] All ML libs available')

## Step 5 — Resume check (which ensembles still need training)

An ensemble is "complete" when all algorithm `.joblib` files + the shared meta JSON exist.

In [ ]:
print('[Step 5/8] Checking which ensembles still need training...')
from src.nfl.training.v5.config import STATS_TO_PREDICT, MODEL_TYPES, get_algorithms
from src.nfl.training.v5.models import ensemble_files_complete
from pathlib import Path

missing = []
complete = []
for position, stats in STATS_TO_PREDICT.items():
    algos = get_algorithms(position)
    for stat in stats:
        for mt in MODEL_TYPES:
            if ensemble_files_complete(Path(MODELS_DIR), position, stat, mt, algos):
                complete.append((position, stat, mt))
            else:
                missing.append((position, stat, mt))

print(f'  Complete: {len(complete)} ensembles')
print(f'  Missing:  {len(missing)} ensembles')
for pos, stat, mt in missing[:20]:
    print(f'    {pos:4} / {stat:25} / {mt}')
if len(missing) > 20:
    print(f'    ... +{len(missing)-20} more')

## Step 5b — Print feature column count + algorithms per position

Catches feature/algo drift between Drive code and uploaded parquets before any training starts.

In [ ]:
print('[Step 5b/8] Inspecting feature shape per position...')
from src.nfl.training.v5.data import load_features, apply_history_filter, get_feature_columns
from src.nfl.training.v5.config import TRAIN_SEASONS, get_algorithms

for pos in STATS_TO_PREDICT.keys():
    df = load_features(pos, TRAIN_SEASONS, features_dir=Path(FEATURES_DIR))
    df = apply_history_filter(df)
    fc = get_feature_columns(df, pos)
    print(f'  {pos:4}  rows={len(df):6}  features={len(fc):4}  algos={get_algorithms(pos)}')
print('[Step 5b/8] Inspection complete')

## Step 6 — Train missing ensembles

Per-ensemble flow: load features → walk-forward eval → refit on all data → save .joblib + meta JSON → append summary row.

If Colab disconnects mid-training, just re-run this cell — it picks up from the next missing ensemble.

In [ ]:
print('[Step 6/8] Training missing ensembles...')
import time
from src.nfl.training.v5.train import train_all

# Override base data dir via monkey-patch (training code uses relative paths by default)
import src.nfl.training.v5.data as v5data
import src.nfl.training.v5.train as v5train
v5data._features_dir = lambda: Path(FEATURES_DIR)
v5train._models_dir  = lambda: Path(MODELS_DIR)
v5train._summary_path = lambda: Path(MODELS_DIR) / '_mae_summary.csv'

t0 = time.time()
summary_df = train_all()
elapsed = time.time() - t0
print(f'\n[Step 6/8] Training run complete in {elapsed/60:.1f} min')

## Step 7 — Verify all 54 ensembles present

In [ ]:
print('[Step 7/8] Verifying all 54 ensembles present...')
from pathlib import Path
# Re-import in case the user jumped here without running Step 5 (common after reconnect)
from src.nfl.training.v5.config import STATS_TO_PREDICT, MODEL_TYPES, get_algorithms
from src.nfl.training.v5.models import ensemble_files_complete

missing_after = []
complete_after = 0
for position, stats in STATS_TO_PREDICT.items():
    algos = get_algorithms(position)
    for stat in stats:
        for mt in MODEL_TYPES:
            if ensemble_files_complete(Path(MODELS_DIR), position, stat, mt, algos):
                complete_after += 1
            else:
                missing_after.append((position, stat, mt))

joblib_count = len(list(Path(MODELS_DIR).glob('*.joblib')))
meta_count = len(list(Path(MODELS_DIR).glob('*_meta.json')))
print(f'  Complete ensembles: {complete_after}/54')
print(f'  .joblib files:      {joblib_count}  (expect 174)')
print(f'  meta JSON files:    {meta_count}  (expect 54)')
if missing_after:
    print(f'\n  WARNING: Missing {len(missing_after)} ensembles:')
    for m in missing_after[:10]:
        print(f'    {m}')

## Step 8 — Print MAE / AUC summary

In [ ]:
print('[Step 8/8] MAE / AUC summary table...')
import pandas as pd
from pathlib import Path

summary_path = Path(MODELS_DIR) / '_mae_summary.csv'
if not summary_path.exists():
    print('  No summary CSV found — training may have errored out.')
else:
    df = pd.read_csv(summary_path)
    stat_rows = df[df['model_type'] == 'stat'][['position','stat','mae_v5','n_eval_predictions','n_train_rows']].copy()
    stat_rows = stat_rows.sort_values(['position','stat'])
    print('\n=== StatPredictor MAE ===')
    print(stat_rows.to_string(index=False))
    pob_cols = ['position','stat','accuracy','auc','pos_class_frac','degenerate_pob','n_eval_predictions']
    pob_rows = df[df['model_type'] == 'pob'][[c for c in pob_cols if c in df.columns]].copy()
    pob_rows = pob_rows.sort_values(['position','stat'])
    print('\n=== POB Accuracy / AUC ===')
    print(pob_rows.to_string(index=False))
    if 'degenerate_pob' in df.columns and (df['degenerate_pob'] == 1).any():
        n_deg = int((df['degenerate_pob'] == 1).sum())
        print(f'\n  WARNING: {n_deg} POB ensemble(s) flagged degenerate_pob=1 (class balance <5% or >95%) — accuracy not meaningful for those rows')

print('\n[Step 8/8] Training pipeline complete.')
print('\nReport back: paste the MAE table above + total file count from Step 7.')